# 07b - Checks

Lightweight validation of the `run_id`/line-split assignment built in `07`. Reads the reduced `7_df_departure_ready.parquet`.

In [1]:
import pandas as pd
import numpy as np

BASEPATH = "../data"

df = pd.read_parquet(f"{BASEPATH}/7_df_departure_ready.parquet")
print(df.shape)

(1551754, 106)


## 1. `run_id` monotonicity per (service_date, train_number)

Each `run_id` sequence should start at 1 and increase by exactly 1 per run within a
`(service_date, train_number)` group. Confirms the `cumsum()` assignment in `07` didn't skip or
duplicate a run anywhere.

In [2]:
run_id_check = (
    df.sort_values(["service_date", "train_number", "run_id"])
    .groupby(["service_date", "train_number"])["run_id"]
    .agg(list)
)

def is_clean_sequence(ids):
    return ids == list(range(1, len(ids) + 1))

bad = run_id_check[~run_id_check.apply(is_clean_sequence)]
print("(service_date, train_number) groups with a non-clean run_id sequence:",
      f"{len(bad):,} / {len(run_id_check):,}")
if len(bad):
    print(bad.head())

(service_date, train_number) groups with a non-clean run_id sequence: 0 / 998,344


## 2. Through-running line-split rate

After the multi-run/through-run fix, we saw **54.6%** of train-days show more than one distinct `line` among their terminal rows. This rebuild uses the identical gap+line-change rule, so it should reproduce the same rate.

In [3]:
lines_per_train_day = df.groupby(["service_date", "train_number"])["line"].nunique()
multi_line_share = (lines_per_train_day > 1).mean()
print(f"Share of (service_date, train_number) combos with >1 distinct line: {multi_line_share:.1%}")

Share of (service_date, train_number) combos with >1 distinct line: 54.6%


## 3. Turnaround-gap distribution between consecutive runs

Gap between one run's scheduled departure and the next run's on same `(service_date, train_number)`.
Would expect/hope for small/near-zero gaps for multi-run handoffs (line change, no layover) and a cluster at/above my 90-minute threshold for independently-started new runs.

In [4]:
df_sorted = df.sort_values(["service_date", "train_number", "sched_origin_sec"])
next_gap_sec = (
    df_sorted.groupby(["service_date", "train_number"])["sched_origin_sec"]
    .diff(periods = -1) * -1
)
print(next_gap_sec.describe())
print()
print("gaps under 5 minutes (likely multi-run handoffs):", (next_gap_sec < 300).sum())
print("gaps at/above 90 min threshold:", (next_gap_sec >= 5400).sum())

count    553397.000000
mean       2611.088042
std        1030.811624
min          -0.000000
25%        1740.000000
50%        2580.000000
75%        3240.000000
max        5820.000000
Name: sched_origin_sec, dtype: float64

gaps under 5 minutes (likely multi-run handoffs): 7833
gaps at/above 90 min threshold: 1192


## 4. Headway feature coverage

`sched_headway_prior_min`/`sched_headway_next_min` should be NA only for the first/last scheduled run
of each `(service_date, line, direction_id)` group. Everything else should have a real value

In [5]:
print("sched_headway_prior_min missing:", df["sched_headway_prior_min"].isna().mean())
print("sched_headway_next_min missing:", df["sched_headway_next_min"].isna().mean())
print()
print(df[["sched_headway_prior_min", "sched_headway_next_min"]].describe())

sched_headway_prior_min missing: 0.0484709560922672
sched_headway_next_min missing: 0.0484709560922672

       sched_headway_prior_min  sched_headway_next_min
count             1.476539e+06            1.476539e+06
mean              5.095026e+01            5.095026e+01
std               3.186911e+01            3.186911e+01
min               0.000000e+00           -0.000000e+00
25%               3.000000e+01            3.000000e+01
50%               5.300000e+01            5.300000e+01
75%               6.000000e+01            6.000000e+01
max               1.180000e+03            1.180000e+03
